In [2]:
%conda install -c conda-forge uncertainties

Channels:
 - conda-forge
 - defaults
Platform: osx-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 25.7.0

Please update conda by running

    $ conda update -n base -c defaults conda



# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.


In [1]:
# Testing against Julia module to see where errors are
import scipy.optimize as opt
import numpy as np
import uncertainties.unumpy as unp
from types import SimpleNamespace

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [60]:
# Helper funcitons from CBsys that are needed to test the calculations
class Bunch(dict):
    def __init__(self, *args, **kwds):
        super(Bunch, self).__init__(*args, **kwds)
        self.__dict__ = self

def noms(*it):
    """
    Return nominal_values for provided objects.

    Parameters
    ----------
    *it : n objects
    """
    return [unp.nominal_values(i) for i in it]

def calc_fH(TempK, Sal):
    # Same as CO2SYS
    # Takahashi et al, Chapter 3 in GEOSECS Pacific Expedition,
    # v. 3, 1982 (p. 80)

    a, b, c, d = (1.2948, -2.036e-3, 4.607e-4, -1.475e-6)
    return a + b * TempK + (c + d * TempK) * Sal ** 2
    

def maxSize(*it):
    """
    Calculate maximum size of provided items.

    Parameters
    ----------
    *it : objects
        Items of various sizes. Only sizes
        of iterables are returned.

    Returns
    -------
    size of largest object (int).
    """
    m = set()
    for i in it:
        try:
            m.add(np.size(i))
        except TypeError:
            pass
    if np.size(m) > 0:
        return max(m)
    else:
        return 1


def maxShape(*it):
    """
    Returns the shape of the largest array.
    """
    size = 0
    shape = None
    for i in it:
        i = np.asanyarray(i)
        if i.size > size:
            size = i.size
            shape = i.shape
    return shape


def cast_array(*it):
    """
    Recasts inputs into array of shape (len(it), maxSize(*it))
    """
    new = np.zeros((len(it), maxSize(*it)))
    for i, t in enumerate(it):
        new[i, :] = np.ravel(t)
    return new


def NnotNone(*it):
    """
    Returns the number of elements of it tha are not None.

    Parameters
    ----------
    it : iterable
        iterable of elements that are either None, or not None

    Returns
    -------
    int
    """
    return sum([i is not None for i in it])

In [5]:
def _zero_wrapper(ps, fn, bounds=(10 ** -14, 10 ** -1)):
    """
    Wrapper to handle zero finders.
    """
    try:
        return opt.brentq(fn, *bounds, args=tuple(ps), xtol=1e-16)
        # brentq is ~100 times faster.
    except ValueError:
        return opt.fsolve(fn, 1, args=tuple(ps))[0]
        # but can be fragile if limits aren't right.

In [7]:
# Function 1

def CO2_pH(CO2, pH, Ks):
    """
    Returns DIC
    """
    h = 10.0**-pH
    return CO2 * (1 + Ks.K1 / h + Ks.K1 * Ks.K2 / h ** 2)

In [22]:
# Test 1
Ks = SimpleNamespace(
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667) # From Leuker for T ~ 25, S ~ 35
)

CO2_val = 10 * 10**(-6)  # 10 uM
pH_val = 8.1

DIC = CO2_pH(CO2_val, pH_val, Ks)

print(f"Calculated DIC: {DIC:.6e} M (or {DIC * 1e6:.2f} µM)")

Calculated DIC: 2.043994e-03 M (or 2043.99 µM)


In [9]:
# Function 2

def CO2_HCO3(CO2, HCO3, Ks):
    """
    Returns H
    """
    CO2, HCO3 = noms(CO2, HCO3)  # get nominal values of inputs
    par = cast_array(CO2, HCO3, Ks.K1, Ks.K2)  # cast parameters into array
    shape = maxShape(CO2, HCO3, Ks.K1, Ks.K2)  # get shape of output


    return np.apply_along_axis(_zero_wrapper, 0, par, fn=zero_CO2_HCO3).reshape(shape)

def zero_CO2_HCO3(h, CO2, HCO3, K1, K2):
    # Roots: two negative, one positive - use positive.
    LH = CO2 * (h ** 2 + K1 * h + K1 * K2)
    RH = HCO3 * (h ** 2 + h ** 3 / K1 + K2 * h)
    return LH - RH

In [26]:
# Test 2
Ks = SimpleNamespace(
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667) # From Leuker for T ~ 25, S ~ 35
)

CO2_val = 10 * 10**(-6)  # 10 uM
HCO3_val = 0.00175

H = CO2_HCO3(CO2_val, HCO3_val, Ks)

print(f"Calculated [H]: {H} M")

Calculated [H]: 8.127593069533137e-09 M


In [11]:
# Function 3

def CO2_CO3(CO2, CO3, Ks):
    """
    Returns H
    """
    CO2, CO3 = noms(CO2, CO3)
    par = cast_array(CO2, CO3, Ks.K1, Ks.K2)  # cast parameters into array
    shape = maxShape(CO2, CO3, Ks.K1, Ks.K2)  # get shape of output


    return np.apply_along_axis(_zero_wrapper, 0, par, fn=zero_CO2_CO3).reshape(shape)


def zero_CO2_CO3(h, CO2, CO3, K1, K2):
    # Roots: one positive, three negative. Use positive.
    LH = CO2 * (h ** 2 + K1 * h + K1 * K2)
    RH = CO3 * (h ** 2 + h ** 3 / K2 + h ** 4 / (K1 * K2))
    return LH - RH

In [30]:
# Test 3
Ks = SimpleNamespace(
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667) # From Leuker for T ~ 25, S ~ 35
)

CO2_val = 10 * 10**(-6)  # 10 uM
CO3_val = 0.0002625

H = CO2_CO3(CO2_val, CO3_val, Ks)

print(f"Calculated [H]: {H} M")

Calculated [H]: 7.64865982634791e-09 M


In [13]:
# Function 4

def CO2_TA(CO2, TA, BT, PT, SiT, ST, FT, Ks):
    """
    Returns pH

    Taken from matlab CO2SYS
    """
    
    fCO2 = CO2 / Ks.K0
    L = maxShape(TA, CO2, BT, PT, SiT, ST, FT, Ks.K1)
    if len(L) == 0:
        L = (1,)
        
    pHguess = 8.0
    pHtol = 0.0000001
    pHx = np.full(L, pHguess)
    deltapH = np.array(pHtol + 1, ndmin=1)
    ln10 = np.log(10)

    while np.any(abs(deltapH) > pHtol):
        H = 10 ** -pHx
        HCO3 = Ks.K0 * Ks.K1 * fCO2 / H
        CO3 = Ks.K0 * Ks.K1 * Ks.K2 * fCO2 / H ** 2
        CAlk = HCO3 + 2 * CO3
        BAlk = BT * Ks.KB / (Ks.KB + H)
        OH = Ks.KW / H
        PhosTop = Ks.KP1 * Ks.KP2 * H + 2 * Ks.KP1 * Ks.KP2 * Ks.KP3 - H ** 3
        PhosBot = (
            H ** 3 + Ks.KP1 * H ** 2 + Ks.KP1 * Ks.KP2 * H + Ks.KP1 * Ks.KP2 * Ks.KP3
        )
        PAlk = PT * PhosTop / PhosBot
        SiAlk = SiT * Ks.KSi / (Ks.KSi + H)
        # positive
        Hfree = H / (1 + ST / Ks.KS)
        HSO4 = ST / (1 + Ks.KS / Hfree)
        HF = FT / (1 + Ks.KF / Hfree)

        Residual = TA - CAlk - BAlk - OH - PAlk - SiAlk + Hfree + HSO4 + HF
        Slope = ln10 * (HCO3 + 4.0 * CO3 + BAlk * H / (Ks.KB + H) + OH + H)
        deltapH = Residual / Slope

        while np.any(abs(deltapH) > 1):
            FF = abs(deltapH) > 1
            deltapH[FF] = deltapH[FF] / 2

        pHx += deltapH

    return pHx

In [34]:
# Test 4

Ks = SimpleNamespace(
    K0 = 0.03,            # Henry's constant for CO2
    K1 = 10.0**(-5.847),   # Carbonate 1
    K2 = 10.0**(-8.966),   # Carbonate 2
    KB = 10.0**(-8.6),     # Boric acid
    KW = 10.0**(-13.2),    # Water
    KP1 = 10.0**(-1.6),    # Phosphoric acid 1
    KP2 = 10.0**(-6.0),    # Phosphoric acid 2
    KP3 = 10.0**(-8.6),    # Phosphoric acid 3
    KSi = 10.0**(-9.5),    # Silicic acid
    KS = 10.0**(-1.0),     # Bisulfate
    KF = 10.0**(-2.6)      # Hydrogen fluoride
)

CO2_val = 10 * 10**(-6)  # 10 uM
TA_val = 2350 * 10**(-6) # 2350 uM
BT_val  = 416.0e-6    # Total Boron ~ 416 µM
PT_val  = 1.0e-6      # Total Phosphate ~ 1 µM
SiT_val = 10.0e-6     # Total Silicate ~ 10 µM
ST_val  = 0.0282      # Total Sulfate ~ 28.2 mM
FT_val  = 6.8e-5      # Total Fluoride ~ 68 µM

pH = CO2_TA(CO2_val, TA_val, BT_val, PT_val, SiT_val, ST_val, FT_val, Ks)

print(f"Calculated pH: {pH}")

Calculated pH: [8.09420997]


In [15]:
# Funciton 5
def CO2_DIC(CO2, DIC, Ks):
    """
    Returns H
    """
    CO2, DIC = noms(CO2, DIC)  # get nominal values of inputs
    par = cast_array(CO2, DIC, Ks.K1, Ks.K2)  # cast parameters into array
    shape = maxShape(CO2, DIC, Ks.K1, Ks.K2)  # get shape of output

    return np.apply_along_axis(_zero_wrapper, 0, par, fn=zero_CO2_DIC).reshape(shape)


def zero_CO2_DIC(h, CO2, DIC, K1, K2):
    # Roots: one positive, one negative. Use positive.
    LH = DIC * h ** 2
    RH = CO2 * (h ** 2 + K1 * h + K1 * K2)
    return LH - RH

In [38]:
# Test 5
Ks = SimpleNamespace(
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667) # From Leuker for T ~ 25, S ~ 35
)

CO2_val = 10 * 10**(-6)  # 10 uM
DIC_val = 0.002

H = CO2_DIC(CO2_val, DIC_val, Ks)

print(f"Calculated [H]: {H} M")

Calculated [H]: 8.100083601585328e-09 M


In [17]:
# Function 6
def pH_HCO3(pH, HCO3, Ks):
    """
    Returns DIC
    """
    h = 10.0**-pH
    return HCO3 * (1 + h / Ks.K1 + Ks.K2 / h)


In [42]:
# Test 6
Ks = SimpleNamespace(
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667) # From Leuker for T ~ 25, S ~ 35
)

HCO3_val = .00175
pH_val = 8.1

DIC = pH_HCO3(pH_val, HCO3_val, Ks)

print(f"Calculated DIC: {DIC} M")

Calculated DIC: 0.001997642338982039 M


In [19]:
# Function 7
def pH_CO3(pH, CO3, Ks):
    """
    Returns DIC
    """
    h = 10.0**-pH
    return CO3 * (1 + h / Ks.K2 + h ** 2 / (Ks.K1 * Ks.K2))


In [14]:
# Test 7
Ks = SimpleNamespace(
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667) # From Leuker for T ~ 25, S ~ 35
)

CO3_val = .0002625
pH_val = 8.1

DIC = pH_CO3(pH_val, CO3_val, Ks)

print(f"Calculated DIC: {DIC} M")

Calculated DIC: 0.002204494365481448 M


In [21]:
# Function 8
def pH_TA(pH, TA, BT, PT, SiT, ST, FT, Ks):
    """
    Returns DIC

    Taken directly from MATLAB CO2SYS.
    """
    H = 10 ** -pH
    # negative alk
    BAlk = BT * Ks.KB / (Ks.KB + H)
    OH = Ks.KW / H
    PhosTop = Ks.KP1 * Ks.KP2 * H + 2 * Ks.KP1 * Ks.KP2 * Ks.KP3 - H ** 3
    PhosBot = H ** 3 + Ks.KP1 * H ** 2 + Ks.KP1 * Ks.KP2 * H + Ks.KP1 * Ks.KP2 * Ks.KP3
    PAlk = PT * PhosTop / PhosBot
    SiAlk = SiT * Ks.KSi / (Ks.KSi + H)
    # positive alk
    Hfree = H / (1 + ST / Ks.KS)
    HSO4 = ST / (1 + Ks.KS / Hfree)
    HF = FT / (1 + Ks.KF / Hfree)
    CAlk = TA - BAlk - OH - PAlk - SiAlk + Hfree + HSO4 + HF

    return CAlk * (H ** 2 + Ks.K1 * H + Ks.K1 * Ks.K2) / (Ks.K1 * (H + 2.0 * Ks.K2))

In [22]:
# Test 8

Ks = SimpleNamespace(
    K0 = 0.03,            # Henry's constant for CO2
    K1 = 10.0**(-5.847),   # Carbonate 1
    K2 = 10.0**(-8.966),   # Carbonate 2
    KB = 10.0**(-8.6),     # Boric acid
    KW = 10.0**(-13.2),    # Water
    KP1 = 10.0**(-1.6),    # Phosphoric acid 1
    KP2 = 10.0**(-6.0),    # Phosphoric acid 2
    KP3 = 10.0**(-8.6),    # Phosphoric acid 3
    KSi = 10.0**(-9.5),    # Silicic acid
    KS = 10.0**(-1.0),     # Bisulfate
    KF = 10.0**(-2.6)      # Hydrogen fluoride
)

pH_val = 8.1
TA_val = 2350 * 10**(-6) # 2350 uM
BT_val  = 416.0e-6    # Total Boron ~ 416 µM
PT_val  = 1.0e-6      # Total Phosphate ~ 1 µM
SiT_val = 10.0e-6     # Total Silicate ~ 10 µM
ST_val  = 0.0282      # Total Sulfate ~ 28.2 mM
FT_val  = 6.8e-5      # Total Fluoride ~ 68 µM

DIC = pH_TA(pH_val, TA_val, BT_val, PT_val, SiT_val, ST_val, FT_val, Ks)

print(f"Calculated DIC: {DIC}")

Calculated DIC: 0.0020105878100041006


In [23]:
# Function 9
def pH_DIC(pH, DIC, Ks):
    """
    Returns CO2
    """
    h = 10.0**-pH
    return DIC / (1 + Ks.K1 / h + Ks.K1 * Ks.K2 / h ** 2)


In [26]:
# Test 9

Ks = SimpleNamespace(
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667) # From Leuker for T ~ 25, S ~ 35
)

DIC_val = .002
pH_val = 8.1

CO2 = pH_DIC(pH_val, DIC_val, Ks)

print(f"Calculated CO2: {CO2} M")

Calculated CO2: 9.784762985024446e-06 M


In [25]:
# Function 10

def HCO3_CO3(HCO3, CO3, Ks):
    """
    Returns H
    """
    HCO3, CO3 = noms(HCO3, CO3)  # get nominal values of inputs
    par = cast_array(HCO3, CO3, Ks.K1, Ks.K2)  # cast parameters into array
    shape = maxShape(HCO3, CO3, Ks.K1, Ks.K2)  # get shape of output
    
    return np.apply_along_axis(_zero_wrapper, 0, par, fn=zero_HCO3_CO3).reshape(shape)


def zero_HCO3_CO3(h, HCO3, CO3, K1, K2):
    # Roots: one pos, two neg. Use pos.
    LH = HCO3 * (h + h ** 2 / K1 + K2)
    RH = CO3 * (h + h ** 2 / K2 + h ** 3 / (K1 * K2))
    return LH - RH


In [30]:
# Test 10

Ks = SimpleNamespace(
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667) # From Leuker for T ~ 25, S ~ 35
)

HCO3_val = 0.00175
CO3_val = 0.0002625

H = HCO3_CO3(HCO3_val, CO3_val, Ks)

print(f"Calculated [H]: {H} M")

Calculated [H]: 7.19794859800552e-09 M


In [27]:
# Function 11

def HCO3_TA(HCO3, TA, BT, Ks):
    """
    Returns H
    """
    HCO3, TA, BT = noms(HCO3, TA, BT)  # get nominal values of inputs
    par = cast_array(
        HCO3, TA, BT, Ks.K1, Ks.K2, Ks.KB, Ks.KW
    )  # cast parameters into array
    shape = maxShape(HCO3, TA, BT, Ks.K1, Ks.K2, Ks.KB, Ks.KW)  # get shape of output

    return np.apply_along_axis(_zero_wrapper, 0, par, fn=zero_HCO3_TA).reshape(shape)


def zero_HCO3_TA(h, HCO3, TA, BT, K1, K2, KB, KW):
    # Roots: one pos, four neg. Use pos.
    LH = TA * (KB + h) * (h ** 3 + K1 * h ** 2 + K1 * K2 * h)
    RH = (
        HCO3
        * (h + h ** 2 / K1 + K2)
        * ((KB + 2 * K2) * K1 * h + 2 * KB * K1 * K2 + K1 * h ** 2)
    ) + (
        (h ** 2 + K1 * h + K1 * K2)
        * (KB * BT * h + KW * KB + KW * h - KB * h ** 2 - h ** 3)
    )
    return LH - RH

In [38]:
# Test 11

Ks = SimpleNamespace(
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667), # From Leuker for T ~ 25, S ~ 35
    KB = 10.0**(-8.6),
    KW = 10.0**(-13.2)
)

HCO3_val = 0.00175
TA_val = 0.002350
BT_val  = 416.0e-6

H = HCO3_TA(HCO3_val, TA_val, BT_val, Ks)

print(f"Calculated [H]: {H} M")

Calculated [H]: 7.71717227447345e-09 M


In [29]:
# Function 12

def HCO3_DIC(HCO3, DIC, Ks):
    """
    Returns H
    """
    HCO3, DIC = noms(HCO3, DIC)  # get nominal values of inputs
    par = cast_array(HCO3, DIC, Ks.K1, Ks.K2)  # cast parameters into array
    shape = maxShape(HCO3, DIC, Ks.K1, Ks.K2)  # get shape of output
    
    return np.apply_along_axis(_zero_wrapper, 0, par, fn=zero_HCO3_DIC).reshape(shape)


def zero_HCO3_DIC(h, HCO3, DIC, K1, K2):
    # Roots: two pos. Use smaller.
    LH = HCO3 * (h + h ** 2 / K1 + K2)
    RH = h * DIC
    return LH - RH


In [54]:
# Test 12

Ks = SimpleNamespace(
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667) # From Leuker for T ~ 25, S ~ 35
)

HCO3_val = 0.00175
DIC_val = 0.002

H = HCO3_DIC(HCO3_val, DIC_val, Ks)

print(f"Calculated H: {H}M")


Calculated H: 1.953277731161216e-07M


In [31]:
# Function 13

def CO3_TA(CO3, TA, BT, Ks):
    """
    Returns H
    """
    CO3, TA, BT = noms(CO3, TA, BT)  # get nominal values of inputs
    par = cast_array(
        CO3, TA, BT, Ks.K1, Ks.K2, Ks.KB, Ks.KW
    )  # cast parameters into array
    shape = maxShape(CO3, TA, BT, Ks.K1, Ks.K2, Ks.KB, Ks.KW)  # get shape of output
    
    return np.apply_along_axis(_zero_wrapper, 0, par, fn=zero_CO3_TA).reshape(shape)


def zero_CO3_TA(h, CO3, TA, BT, K1, K2, KB, KW):
    # Roots: three neg, two pos. Use larger pos.
    LH = TA * (KB + h) * (h ** 3 + K1 * h ** 2 + K1 * K2 * h)
    RH = (
        CO3
        * (h + h ** 2 / K2 + h ** 3 / (K1 * K2))
        * (K1 * h ** 2 + K1 * h * (KB + 2 * K2) + 2 * KB * K1 * K2)
    ) + (
        (h ** 2 + K1 * h + K1 * K2)
        * (KB * BT * h + KW * KB + KW * h - KB * h ** 2 - h ** 3)
    )
    return LH - RH

In [64]:
# Test 13

Ks = SimpleNamespace(
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667), # From Leuker for T ~ 25, S ~ 35
    KB = 10.0**(-8.6),
    KW = 10.0**(-13.2)
)

CO3_val = 0.0002625
TA_val = 0.002350
BT_val  = 416.0e-6

H = CO3_TA(CO3_val, TA_val, BT_val, Ks)

print(f"Calculated [H]: {H} M")

Calculated [H]: 7.01850880499085e-09 M


In [33]:
# Function 14

def CO3_DIC(CO3, DIC, Ks):
    """
    Returns H
    """
    CO3, DIC = noms(CO3, DIC)  # get nominal values of inputs
    par = cast_array(CO3, DIC, Ks.K1, Ks.K2)  # cast parameters into array
    shape = maxShape(CO3, DIC, Ks.K1, Ks.K2)  # get shape of output

    return np.apply_along_axis(_zero_wrapper, 0, par, fn=zero_CO3_DIC).reshape(shape)


def zero_CO3_DIC(h, CO3, DIC, K1, K2):
    # Roots: one pos, one neg. Use neg.
    LH = CO3 * (1 + h / K2 + h ** 2 / (K1 * K2))
    RH = DIC
    return LH - RH

In [70]:
# Test 14

Ks = SimpleNamespace(
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667) # From Leuker for T ~ 25, S ~ 35
)

HCO3_val = 0.0002625
DIC_val = 0.002

H = CO3_DIC(CO3_val, DIC_val, Ks)

print(f"Calculated H: {H}M")

Calculated H: 7.1109830637671875e-09M


In [35]:
# Function 15

def TA_DIC(TA, DIC, BT, PT, SiT, ST, FT, Ks):
    """
    Returns pH

    Taken directly from MATLAB CO2SYS.
    """
    # determine shape of input
    L = maxShape(TA, DIC, BT, PT, SiT, ST, FT, Ks.K1)
    if len(L) == 0:
        L = (1,)
        
    pHguess = 8.0
    pHtol = 0.00000001
    pHx = np.full(L, pHguess)
    deltapH = np.array(pHtol + 1, ndmin=1)
    ln10 = np.log(10)

    while np.any(abs(deltapH) > pHtol):
        H = 10 ** -pHx
        # negative
        Denom = H ** 2 + Ks.K1 * H + Ks.K1 * Ks.K2
        CAlk = DIC * Ks.K1 * (H + 2 * Ks.K2) / Denom
        BAlk = BT * Ks.KB / (Ks.KB + H)
        OH = Ks.KW / H
        PhosTop = Ks.KP1 * Ks.KP2 * H + 2 * Ks.KP1 * Ks.KP2 * Ks.KP3 - H ** 3
        PhosBot = (
            H ** 3 + Ks.KP1 * H ** 2 + Ks.KP1 * Ks.KP2 * H + Ks.KP1 * Ks.KP2 * Ks.KP3
        )
        PAlk = PT * PhosTop / PhosBot
        SiAlk = SiT * Ks.KSi / (Ks.KSi + H)
        # positive
        Hfree = H / (1 + ST / Ks.KS)
        HSO4 = ST / (1 + Ks.KS / Hfree)
        HF = FT / (1 + Ks.KF / Hfree)

        Residual = TA - CAlk - BAlk - OH - PAlk - SiAlk + Hfree + HSO4 + HF

        Slope = ln10 * (
            DIC * Ks.K1 * H * (H ** 2 + Ks.K1 * Ks.K2 + 4 * H * Ks.K2) / Denom / Denom
            + BAlk * H / (Ks.KB + H)
            + OH
            + H
        )
        deltapH = Residual / Slope

        while np.any(abs(deltapH) > 1):
            FF = abs(deltapH) > 1
            deltapH[FF] = deltapH[FF] / 2

        pHx += deltapH

    return pHx


def zero_TA_DIC(h, TA, DIC, BT, K1, K2, KB, KW):
    # Roots: one pos, four neg. Use pos.
    LH = DIC * (KB + h) * (K1 * h ** 2 + 2 * K1 * K2 * h)
    RH = (TA * (KB + h) * h - KB * BT * h - KW * (KB + h) + (KB + h) * h ** 2) * (
        h ** 2 + K1 * h + K1 * K2
    )
    return LH - RH

In [76]:
# Test 15

Ks = SimpleNamespace(
    K0 = 0.03,            # Henry's constant for CO2
    K1 = 10.0**(-5.847),   # Carbonate 1
    K2 = 10.0**(-8.966),   # Carbonate 2
    KB = 10.0**(-8.6),     # Boric acid
    KW = 10.0**(-13.2),    # Water
    KP1 = 10.0**(-1.6),    # Phosphoric acid 1
    KP2 = 10.0**(-6.0),    # Phosphoric acid 2
    KP3 = 10.0**(-8.6),    # Phosphoric acid 3
    KSi = 10.0**(-9.5),    # Silicic acid
    KS = 10.0**(-1.0),     # Bisulfate
    KF = 10.0**(-2.6)      # Hydrogen fluoride
)

pH_val = 8.1
TA_val = 2350 * 10**(-6) # 2350 uM
BT_val  = 416.0e-6    # Total Boron ~ 416 µM
PT_val  = 1.0e-6      # Total Phosphate ~ 1 µM
SiT_val = 10.0e-6     # Total Silicate ~ 10 µM
ST_val  = 0.0282      # Total Sulfate ~ 28.2 mM
FT_val  = 6.8e-5      # Total Fluoride ~ 68 µM

pH = TA_DIC(TA_val, DIC_val, BT_val, PT_val, SiT_val, ST_val, FT_val, Ks)

print(f"Calculated pH: {pH}")


Calculated pH: [8.11651141]


In [37]:
# Function 1.1.9
def cCO2(H, DIC, Ks):
    """
    Returns CO2
    """
    return DIC / (1 + Ks.K1 / H + Ks.K1 * Ks.K2 / H ** 2)


In [86]:
# Test 1.1.9
Ks = SimpleNamespace(
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667) # From Leuker for T ~ 25, S ~ 35
)

H_val = 10**(-8.1)
DIC_val = 0.002

H = cCO2(H_val, DIC_val, Ks)

print(f"Calculated CO2: {CO2}M")

Calculated CO2: 9.784762985024446e-06M


In [39]:
# Function 1.1.10
def cHCO3(H, DIC, Ks):
    """
    Returns HCO3
    """
    return DIC / (1 + H / Ks.K1 + Ks.K2 / H)


In [90]:
# Test 1.1.10
Ks = SimpleNamespace(
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667) # From Leuker for T ~ 25, S ~ 35
)

H_val = 10**(-8.1)
DIC_val = 0.002

HCO3 = cHCO3(H_val, DIC_val, Ks)

print(f"Calculated HCo3: {HCO3}M")

Calculated HCo3: 0.0017520653881332603M


In [41]:
# Function 1.1.11

def cCO3(H, DIC, Ks):
    """
    Returns CO3
    """
    return DIC / (1 + H / Ks.K2 + H ** 2 / (Ks.K1 * Ks.K2))



In [96]:
# Test 1.1.11

Ks = SimpleNamespace(
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667) # From Leuker for T ~ 25, S ~ 35
)

H_val = 10**(-8.1)
DIC_val = 0.002

CO3 = cCO3(H_val, DIC_val, Ks)

print(f"Calculated CO3: {CO3} M")


Calculated CO3: 0.00023814984888171544 M


In [43]:
# Function 1.5.80
def cTA(H, DIC, BT, PT, SiT, ST, FT, Ks, mode="multi"):
    """
    Calculate Alkalinity. H is on Total scale.

    Returns
    -------
    If mode == 'multi' returns TA, CAlk, PAlk, SiAlk, OH
    else: returns TA
    """
    # negative
    Denom = H ** 2 + Ks.K1 * H + Ks.K1 * Ks.K2
    CAlk = DIC * Ks.K1 * (H + 2 * Ks.K2) / Denom
    BAlk = BT * Ks.KB / (Ks.KB + H)
    OH = Ks.KW / H
    PhosTop = Ks.KP1 * Ks.KP2 * H + 2 * Ks.KP1 * Ks.KP2 * Ks.KP3 - H ** 3
    PhosBot = H ** 3 + Ks.KP1 * H ** 2 + Ks.KP1 * Ks.KP2 * H + Ks.KP1 * Ks.KP2 * Ks.KP3
    PAlk = PT * PhosTop / PhosBot
    SiAlk = SiT * Ks.KSi / (Ks.KSi + H)
    # positive
    Hfree = H / (1 + ST / Ks.KS)
    HSO4 = ST / (1 + Ks.KS / Hfree)
    HF = FT / (1 + Ks.KF / Hfree)

    TA = CAlk + BAlk + OH + PAlk + SiAlk - Hfree - HSO4 - HF

    if mode == "multi":
        return TA, CAlk, BAlk, PAlk, SiAlk, OH, Hfree, HSO4, HF
    else:
        return TA

In [80]:
# Test 1.5.80

Ks = SimpleNamespace(
    K0 = 0.03,            # Henry's constant for CO2
    K1 = 10.0**(-5.847),   # Carbonate 1
    K2 = 10.0**(-8.966),   # Carbonate 2
    KB = 10.0**(-8.6),     # Boric acid
    KW = 10.0**(-13.2),    # Water
    KP1 = 10.0**(-1.6),    # Phosphoric acid 1
    KP2 = 10.0**(-6.0),    # Phosphoric acid 2
    KP3 = 10.0**(-8.6),    # Phosphoric acid 3
    KSi = 10.0**(-9.5),    # Silicic acid
    KS = 10.0**(-1.0),     # Bisulfate
    KF = 10.0**(-2.6)      # Hydrogen fluoride
)

H_val = 10**(-8.1)
DIC_val = 0.002
BT_val  = 416.0e-6    # Total Boron ~ 416 µM
PT_val  = 1.0e-6      # Total Phosphate ~ 1 µM
SiT_val = 10.0e-6     # Total Silicate ~ 10 µM
ST_val  = 0.0282      # Total Sulfate ~ 28.2 mM
FT_val  = 6.8e-5      # Total Fluoride ~ 68 µM

TA = cTA(H_val, DIC_val, BT_val, PT_val, SiT_val, ST_val, FT_val, Ks)

print(f"Calculated TA: {TA}")

Calculated TA: (0.0023382014457705236, 0.002228705317701652, 9.99452785144495e-05, 1.2328131840987503e-06, 3.828650388225471e-07, 7.943282347242822e-06, 6.1960080711722476e-09, 1.7472741678093253e-09, 1.6773350270791072e-10)


In [45]:
# Function C.4.14
def fCO2_to_CO2(fCO2, Ks):
    """
    Calculate CO2 from fCO2
    """
    return fCO2 * Ks.K0

In [110]:
# Test C.4.14

Ks = SimpleNamespace(
    K0 = 0.03,
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667) # From Leuker for T ~ 25, S ~ 35
)

fCO2 = 343.2

CO2 = fCO2_to_CO2(fCO2, Ks)

print(f"Calculated CO2: {CO2} M")

Calculated CO2: 10.296 M


In [47]:
# Function C.4.14

def CO2_to_fCO2(CO2, Ks):
    """
    Calculate fCO2 from CO2
    """
    return CO2 / Ks.K0

In [114]:
# Test C.4.14
Ks = SimpleNamespace(
    K0 = 0.03,
    K1 = 10**(-5.847), # From Leuker for T ~ 25, S ~ 35
    K2 = 10**(-8.9667) # From Leuker for T ~ 25, S ~ 35
)

CO2 = 10*10**(-6)

fCO2 = CO2_to_fCO2(CO2, Ks)

print(f"Calculated fCO2: {fCO2} M")

Calculated fCO2: 0.0003333333333333333 M


In [49]:
# Function pCO2 to fCO2

def pCO2_to_fCO2(pCO2, Tc):
    """
    Calculate fCO2 from pCO2

    Taken from matlab CO2SYS.

    This assumes that the pressure is at one atmosphere, or close to it.
    Otherwise, the Pres term in the exponent affects the results.
    Weiss, R. F., Marine Chemistry 2:203-215, 1974.

    For a mixture of CO2 and air at 1 atm (at low CO2 concentrations)
    Delta and B in cm3/mol
    """
    Tk = Tc + 273.15
    P = 1.01325  # in bar
    RT = 83.1451 * Tk

    a0, a1, a2, a3 = (-1636.75, 12.0408, -3.27957e-2, 3.16528e-05)
    b0, b1 = (57.7, -0.118)

    B = a0 + a1 * Tk + a2 * Tk ** 2 + a3 * Tk ** 3
    delta = b0 + b1 * Tk

    return pCO2 * np.exp(P * (B + 2 * delta) / RT)

In [126]:
# Test pCO2 to fCO2

T_val = 25

pCO2 = 400 * 10**(-6)

fCO2 = pCO2_to_fCO2(pCO2, T_val)

print(f"Calculated fCO2: {fCO2} M")

Calculated fCO2: 0.000398724183476841 M


In [51]:
# Function fCO2 to pCO2

def fCO2_to_pCO2(fCO2, Tc):
    """
    Calculate pCO2 from fCO2

    Taken from matlab CO2SYS.

    This assumes that the pressure is at one atmosphere, or close to it.
    Otherwise, the Pres term in the exponent affects the results.
    Weiss, R. F., Marine Chemistry 2:203-215, 1974.

    For a mixture of CO2 and air at 1 atm (at low CO2 concentrations)
    Delta and B in cm3/mol
    """
    Tk = Tc + 273.15
    P = 1.01325  # in bar
    RT = 83.1451 * Tk

    a0, a1, a2, a3 = (-1636.75, 12.0408, -3.27957e-2, 3.16528e-05)
    b0, b1 = (57.7, -0.118)

    B = a0 + a1 * Tk + a2 * Tk ** 2 + a3 * Tk ** 3
    delta = b0 + b1 * Tk

    return fCO2 / np.exp(P * (B + 2 * delta) / RT)


In [130]:
# Test fCO2 to pCO2

T_val = 25

fCO2 = 343.2

pCO2 = fCO2_to_pCO2(fCO2, T_val)

print(f"Calculated pCO2: {pCO2} M")

Calculated pCO2: 344.29815318180613 M


In [53]:
# Revelle Factor Function

def calc_revelle_factor(TA, DIC, BT, PT, SiT, ST, FT, Ks):
    """
    Calculate Revelle Factor

    (dpCO2 / dDIC)
    """
    dDIC = 1e-6  # (1 umol kg-1)

    pH = TA_DIC(TA=TA, DIC=DIC, BT=BT, PT=PT, SiT=SiT, ST=ST, FT=FT, Ks=Ks)
    fCO2 = cCO2(10.0**-pH, DIC, Ks) / Ks.K0

    # Calculate new fCO2 above and below given value
    pH_hi = TA_DIC(TA=TA, DIC=DIC + dDIC, BT=BT, PT=PT, SiT=SiT, ST=ST, FT=FT, Ks=Ks)
    fCO2_hi = cCO2(10.0**-pH_hi, DIC, Ks) / Ks.K0

    pH_lo = TA_DIC(TA=TA, DIC=DIC - dDIC, BT=BT, PT=PT, SiT=SiT, ST=ST, FT=FT, Ks=Ks)
    fCO2_lo = cCO2(10.0**-pH_lo, DIC, Ks) / Ks.K0

    return (fCO2_hi - fCO2_lo) * DIC / (fCO2 * 2 * dDIC)

In [136]:
# Revelle Factor Test

Ks = SimpleNamespace(
    K0 = 0.03,            # Henry's constant for CO2
    K1 = 10.0**(-5.847),   # Carbonate 1
    K2 = 10.0**(-8.966),   # Carbonate 2
    KB = 10.0**(-8.6),     # Boric acid
    KW = 10.0**(-13.2),    # Water
    KP1 = 10.0**(-1.6),    # Phosphoric acid 1
    KP2 = 10.0**(-6.0),    # Phosphoric acid 2
    KP3 = 10.0**(-8.6),    # Phosphoric acid 3
    KSi = 10.0**(-9.5),    # Silicic acid
    KS = 10.0**(-1.0),     # Bisulfate
    KF = 10.0**(-2.6)      # Hydrogen fluoride
)

TA_val = 0.002350
DIC_val = 0.002
BT_val  = 416.0e-6    # Total Boron ~ 416 µM
PT_val  = 1.0e-6      # Total Phosphate ~ 1 µM
SiT_val = 10.0e-6     # Total Silicate ~ 10 µM
ST_val  = 0.0282      # Total Sulfate ~ 28.2 mM
FT_val  = 6.8e-5      # Total Fluoride ~ 68 µM

RF = calc_revelle_factor(TA_val, DIC_val, BT_val, PT_val, SiT_val, ST_val, FT_val, Ks)

print(f"Calculated RF: {RF}")

Calculated RF: [7.96709466]


In [55]:
# The big boi


def calc_C_species(
    pHtot=None,
    DIC=None,
    CO2=None,
    HCO3=None,
    CO3=None,
    TA=None,
    fCO2=None,
    pCO2=None,
    T_in=None,
    S_in=None,
    BT=None,
    PT=0,
    SiT=0,
    ST=0,
    FT=0,
    Ks=None,
    **kwargs
):
    """
    Calculate all carbon species from minimal input.
    """

    # if fCO2 is given but CO2 is not, calculate CO2
    if CO2 is None:
        if fCO2 is not None:
            CO2 = fCO2_to_CO2(fCO2, Ks)
        elif pCO2 is not None:
            CO2 = fCO2_to_CO2(pCO2_to_fCO2(pCO2, T_in), Ks)

    # Carbon System Calculations (logic from Zeebe & Wolf-Gladrow, Appendix B)
    # 1. CO2 and pH
    if CO2 is not None and pHtot is not None:
        H = 10.0**-pHtot
        DIC = CO2_pH(CO2, pHtot, Ks)
    # 2. CO2 and HCO3
    elif CO2 is not None and HCO3 is not None:
        H = CO2_HCO3(CO2, HCO3, Ks)
        DIC = CO2_pH(CO2, -np.log10(H), Ks)
    # 3. CO2 and CO3
    elif CO2 is not None and CO3 is not None:
        H = CO2_CO3(CO2, CO3, Ks)
        DIC = CO2_pH(CO2, -np.log10(H), Ks)
    # 4. CO2 and TA
    elif CO2 is not None and TA is not None:
        # unit conversion because OH and H wrapped
        # up in TA fns - all need to be in same units.
        pHtot = CO2_TA(CO2=CO2, TA=TA, BT=BT, PT=PT, SiT=SiT, ST=ST, FT=FT, Ks=Ks)
        H = 10.0**-pHtot
        DIC = CO2_pH(CO2, pHtot, Ks)
    # 5. CO2 and DIC
    elif CO2 is not None and DIC is not None:
        H = CO2_DIC(CO2, DIC, Ks)
    # 6. pHtot and HCO3
    elif pHtot is not None and HCO3 is not None:
        H = 10.0**-pHtot
        DIC = pH_HCO3(pHtot, HCO3, Ks)
    # 7. pHtot and CO3
    elif pHtot is not None and CO3 is not None:
        H = 10.0**-pHtot
        DIC = pH_CO3(pHtot, CO3, Ks)
    # 8. pHtot and TA
    elif pHtot is not None and TA is not None:
        H = 10.0**-pHtot
        DIC = pH_TA(pH=pHtot, TA=TA, BT=BT, PT=PT, SiT=SiT, ST=ST, FT=FT, Ks=Ks)
    # 9. pHtot and DIC
    elif pHtot is not None and DIC is not None:
        H = 10.0**-pHtot
    # 10. HCO3 and CO3
    elif HCO3 is not None and CO3 is not None:
        H = HCO3_CO3(HCO3, CO3, Ks)
        DIC = pH_CO3(-np.log10(H), CO3, Ks)
    # 11. HCO3 and TA
    elif HCO3 is not None and TA is not None:
        Warning(
            "Nutrient alkalinity not implemented for this input combination.\nCalculations use only C and B alkalinity."
        )
        H = HCO3_TA(HCO3, TA, BT, Ks)
        DIC = pH_HCO3(-np.log10(H), HCO3, Ks)
    # 12. HCO3 amd DIC
    elif HCO3 is not None and DIC is not None:
        H = HCO3_DIC(HCO3, DIC, Ks)
    # 13. CO3 and TA
    elif CO3 is not None and TA is not None:
        Warning(
            "Nutrient alkalinity not implemented for this input combination.\nCalculations use only C and B alkalinity."
        )
        H = CO3_TA(CO3, TA, BT, Ks)
        DIC = pH_CO3(-np.log10(H), CO3, Ks)
    # 14. CO3 and DIC
    elif CO3 is not None and DIC is not None:
        H = CO3_DIC(CO3, DIC, Ks)
    # 15. TA and DIC
    elif TA is not None and DIC is not None:
        pHtot = TA_DIC(TA=TA, DIC=DIC, BT=BT, PT=PT, SiT=SiT, ST=ST, FT=FT, Ks=Ks)
        H = 10.0**-pHtot

    # The above makes sure that DIC and H are known,
    # this next bit calculates all the missing species
    # from DIC and H.
    if CO2 is None:
        CO2 = cCO2(H, DIC, Ks)
    if fCO2 is None:
        fCO2 = CO2_to_fCO2(CO2, Ks)
    if pCO2 is None:
        pCO2 = fCO2_to_pCO2(fCO2, T_in)
    if HCO3 is None:
        HCO3 = cHCO3(H, DIC, Ks)
    if CO3 is None:
        CO3 = cCO3(H, DIC, Ks)
    # Calculate all elements of Alkalinity
    (TA, CAlk, BAlk, PAlk, SiAlk, OH, Hfree, HSO4, HF) = cTA(
        H=H, DIC=DIC, BT=BT, PT=PT, SiT=SiT, ST=ST, FT=FT, Ks=Ks, mode="multi"
    )

    # if pH not calced yet, calculate on all scales.
    if pHtot is None:
        pHtot = np.array(-np.log10(H), ndmin=1)
    
    FREEtoTOT = -np.log10((1 + ST / Ks.KS))
    SWStoTOT = -np.log10((1 + ST / Ks.KS) / (1 + ST / Ks.KS + FT / Ks.KF))
    fH = calc_fH(T_in + 273.15, S_in)
    
    return Bunch(
        {
            "pHtot": pHtot,
            "pHfree": pHtot - FREEtoTOT,
            "pHsws": pHtot - SWStoTOT,
            "pHNBS": pHtot - SWStoTOT - np.log10(fH),
            "TA": TA,
            "DIC": DIC,
            "CO2": CO2,
            "H": H,
            "HCO3": HCO3,
            "fCO2": fCO2,
            "pCO2": pCO2,
            "CO3": CO3,
            "CAlk": CAlk,
            "BAlk": BAlk,
            "PAlk": PAlk,
            "SiAlk": SiAlk,
            "OH": OH,
            "Hfree": Hfree,
            "HSO4": HSO4,
            "HF": HF,
        }
    )

In [85]:
# pH & TA

# Ks
Ks = SimpleNamespace(
    K0 = 0.03,            # Henry's constant for CO2
    K1 = 10.0**(-5.847),   # Carbonate 1
    K2 = 10.0**(-8.966),   # Carbonate 2
    KB = 10.0**(-8.6),     # Boric acid
    KW = 10.0**(-13.2),    # Water
    KP1 = 10.0**(-1.6),    # Phosphoric acid 1
    KP2 = 10.0**(-6.0),    # Phosphoric acid 2
    KP3 = 10.0**(-8.6),    # Phosphoric acid 3
    KSi = 10.0**(-9.5),    # Silicic acid
    KS = 10.0**(-1.0),     # Bisulfate
    KF = 10.0**(-2.6)      # Hydrogen fluoride
)

# Pass in the two knowns:
results = calc_C_species(
    pHtot=8.1,
    TA=0.002350,
    T_in=25.0, 
    S_in=35.0, 
    BT=416e-6, 
    ST=0.0282, 
    FT=6.8e-5, 
    Ks=Ks
)

# You can now access any calculated variable using dot notation!
print(f"Calculated pH: {results.pHtot}")
print(f"Calculated TA: {results.TA}")
print(f"Calculated DIC: {results.DIC}")
print(f"Calculated CO2: {results.CO2}")
print(f"Calculated HCO3: {results.HCO3}")
print(f"Calculated CO3: {results.CO3}")

Calculated pH: 8.1
Calculated TA: 0.0023500000000000005
Calculated DIC: 0.0020120376905334724
Calculated CO2: 9.841765544618426e-06
Calculated HCO3: 0.001762272299823658
Calculated CO3: 0.00023992362516519613


In [87]:
# pH & DIC

# Pass in the two knowns:
results = calc_C_species(
    pHtot=8.1,
    DIC=0.002,
    T_in=25.0, 
    S_in=35.0, 
    BT=416e-6, 
    ST=0.0282, 
    FT=6.8e-5, 
    Ks=Ks
)

# You can now access any calculated variable using dot notation!
print(f"Calculated pH: {results.pHtot}")
print(f"Calculated TA: {results.TA}")
print(f"Calculated DIC: {results.DIC}")
print(f"Calculated CO2: {results.CO2}")
print(f"Calculated HCO3: {results.HCO3}")
print(f"Calculated CO3: {results.CO3}")

Calculated pH: 8.1
Calculated TA: 0.0023365857675476023
Calculated DIC: 0.002
Calculated CO2: 9.78288388028057e-06
Calculated HCO3: 0.0017517289145377872
Calculated CO3: 0.0002384882015819323


In [89]:
# pH & CO2

# Pass in the two knowns:
results = calc_C_species(
    pHtot=8.1,
    CO2=1 * 10**(-5),
    T_in=25.0, 
    S_in=35.0, 
    BT=416e-6, 
    ST=0.0282, 
    FT=6.8e-5, 
    Ks=Ks
)

# You can now access any calculated variable using dot notation!
print(f"Calculated pH: {results.pHtot}")
print(f"Calculated TA: {results.TA}")
print(f"Calculated DIC: {results.DIC}")
print(f"Calculated CO2: {results.CO2}")
print(f"Calculated HCO3: {results.HCO3}")
print(f"Calculated CO3: {results.CO3}")

Calculated pH: 8.1
Calculated TA: 0.0023860484675549917
Calculated DIC: 0.0020443869358721663
Calculated CO2: 1e-05
Calculated HCO3: 0.0017906058540352913
Calculated CO3: 0.00024378108183687505


In [91]:
# pH & HCO3

# Pass in the two knowns:
results = calc_C_species(
    pHtot=8.1,
    HCO3=0.00175,
    T_in=25.0, 
    S_in=35.0, 
    BT=416e-6, 
    ST=0.0282, 
    FT=6.8e-5, 
    Ks=Ks
)

# You can now access any calculated variable using dot notation!
print(f"Calculated pH: {results.pHtot}")
print(f"Calculated TA: {results.TA}")
print(f"Calculated DIC: {results.DIC}")
print(f"Calculated CO2: {results.CO2}")
print(f"Calculated HCO3: {results.HCO3}")
print(f"Calculated CO3: {results.CO3}")

Calculated pH: 8.1
Calculated TA: 0.0023343860887090335
Calculated DIC: 0.001998026047839436
Calculated CO2: 9.773228407894557e-06
Calculated HCO3: 0.00175
Calculated CO3: 0.00023825281943154144


In [93]:
# pH & CO3
# Pass in the two knowns:

results = calc_C_species(
    pHtot=8.1,
    CO3=0.0002625,
    T_in=25.0, 
    S_in=35.0, 
    BT=416e-6, 
    ST=0.0282, 
    FT=6.8e-5, 
    Ks=Ks
)

# You can now access any calculated variable using dot notation!
print(f"Calculated pH: {results.pHtot}")
print(f"Calculated TA: {results.TA}")
print(f"Calculated DIC: {results.DIC}")
print(f"Calculated CO2: {results.CO2}")
print(f"Calculated HCO3: {results.HCO3}")
print(f"Calculated CO3: {results.CO3}")

Calculated pH: 8.1
Calculated TA: 0.002560979353758375
Calculated DIC: 0.002201366761615823
Calculated CO2: 1.0767857703398438e-05
Calculated HCO3: 0.0019280989039124248
Calculated CO3: 0.0002625


In [95]:
# TA & DIC

# Pass in the two knowns:

results = calc_C_species(
    TA=0.002350,
    DIC=0.002,
    T_in=25.0, 
    S_in=35.0, 
    BT=416e-6, 
    ST=0.0282, 
    FT=6.8e-5, 
    Ks=Ks
)

# You can now access any calculated variable using dot notation!
print(f"Calculated pH: {results.pHtot}")
print(f"Calculated TA: {results.TA}")
print(f"Calculated DIC: {results.DIC}")
print(f"Calculated CO2: {results.CO2}")
print(f"Calculated HCO3: {results.HCO3}")
print(f"Calculated CO3: {results.CO3}")

Calculated pH: [8.11877535]
Calculated TA: [0.00235]
Calculated DIC: 0.002
Calculated CO2: [9.32178386e-06]
Calculated HCO3: [0.00174291]
Calculated CO3: [0.00024777]


In [97]:
# TA & CO2

results = calc_C_species(
    TA=0.002350,
    CO2=0.00001,
    T_in=25.0, 
    S_in=35.0, 
    BT=416e-6, 
    ST=0.0282, 
    FT=6.8e-5, 
    Ks=Ks
)

# You can now access any calculated variable using dot notation!
print(f"Calculated pH: {results.pHtot}")
print(f"Calculated TA: {results.TA}")
print(f"Calculated DIC: {results.DIC}")
print(f"Calculated CO2: {results.CO2}")
print(f"Calculated HCO3: {results.HCO3}")
print(f"Calculated CO3: {results.CO3}")

Calculated pH: [8.09445936]
Calculated TA: [0.00235]
Calculated DIC: [0.00201555]
Calculated CO2: 1e-05
Calculated HCO3: [0.00176791]
Calculated CO3: [0.00023764]


In [99]:
# TA & HCO3

results = calc_C_species(
    TA=0.002350,
    HCO3=0.00175,
    T_in=25.0, 
    S_in=35.0, 
    BT=416e-6, 
    ST=0.0282, 
    FT=6.8e-5, 
    Ks=Ks
)

# You can now access any calculated variable using dot notation!
print(f"Calculated pH: {results.pHtot}")
print(f"Calculated TA: {results.TA}")
print(f"Calculated DIC: {results.DIC}")
print(f"Calculated CO2: {results.CO2}")
print(f"Calculated HCO3: {results.HCO3}")
print(f"Calculated CO3: {results.CO3}")

Calculated pH: [8.11194556]
Calculated TA: 0.002349999835420221
Calculated DIC: 0.0020044051530799783
Calculated CO2: 9.508072610185833e-06
Calculated HCO3: 0.00175
Calculated CO3: 0.00024489708046979277


In [101]:
# TA & CO3
results = calc_C_species(
    TA=0.002350,
    CO3=0.0002625,
    T_in=25.0, 
    S_in=35.0, 
    BT=416e-6, 
    ST=0.0282, 
    FT=6.8e-5, 
    Ks=Ks
)

# You can now access any calculated variable using dot notation!
print(f"Calculated pH: {results.pHtot}")
print(f"Calculated TA: {results.TA}")
print(f"Calculated DIC: {results.DIC}")
print(f"Calculated CO2: {results.CO2}")
print(f"Calculated HCO3: {results.HCO3}")
print(f"Calculated CO3: {results.CO3}")

Calculated pH: [8.15301631]
Calculated TA: 0.0023499998515420797
Calculated DIC: 0.0019774615610605706
Calculated CO2: 8.435225206711706e-06
Calculated HCO3: 0.0017065263358538585
Calculated CO3: 0.0002625


In [103]:
# DIC & CO2

results = calc_C_species(
    DIC=0.002,
    CO2=0.00001,
    T_in=25.0, 
    S_in=35.0, 
    BT=416e-6, 
    ST=0.0282, 
    FT=6.8e-5, 
    Ks=Ks
)

# You can now access any calculated variable using dot notation!
print(f"Calculated pH: {results.pHtot}")
print(f"Calculated TA: {results.TA}")
print(f"Calculated DIC: {results.DIC}")
print(f"Calculated CO2: {results.CO2}")
print(f"Calculated HCO3: {results.HCO3}")
print(f"Calculated CO3: {results.CO3}")

Calculated pH: [8.09143679]
Calculated TA: 0.002330590317181102
Calculated DIC: 0.002
Calculated CO2: 1e-05
Calculated HCO3: 0.0017556453475194224
Calculated CO3: 0.0002343546524756832


In [105]:
# DIC & HCO3

results = calc_C_species(
    DIC=0.002,
    HCO3=0.00175,
    T_in=25.0, 
    S_in=35.0, 
    BT=416e-6, 
    ST=0.0282, 
    FT=6.8e-5, 
    Ks=Ks
)

# You can now access any calculated variable using dot notation!
print(f"Calculated pH: {results.pHtot}")
print(f"Calculated TA: {results.TA}")
print(f"Calculated DIC: {results.DIC}")
print(f"Calculated CO2: {results.CO2}")
print(f"Calculated HCO3: {results.HCO3}")
print(f"Calculated CO3: {results.CO3}")

Calculated pH: [6.70926538]
Calculated TA: 0.0017747848291603662
Calculated DIC: 0.002
Calculated CO2: 0.00024031045389240403
Calculated HCO3: 0.00175
Calculated CO3: 9.689546107602344e-06


In [107]:
# DIC & CO3

results = calc_C_species(
    DIC=0.002,
    CO3=0.0002625,
    T_in=25.0, 
    S_in=35.0, 
    BT=416e-6, 
    ST=0.0282, 
    FT=6.8e-5, 
    Ks=Ks
)

# You can now access any calculated variable using dot notation!
print(f"Calculated pH: {results.pHtot}")
print(f"Calculated TA: {results.TA}")
print(f"Calculated DIC: {results.DIC}")
print(f"Calculated CO2: {results.CO2}")
print(f"Calculated HCO3: {results.HCO3}")
print(f"Calculated CO3: {results.CO3}")

Calculated pH: [8.14737382]
Calculated TA: 0.0023711552985301995
Calculated DIC: 0.002
Calculated CO2: 8.657283926362271e-06
Calculated HCO3: 0.001728842715931684
Calculated CO3: 0.0002625


In [109]:
# CO2 & HCO3

results = calc_C_species(
    CO2=0.00001,
    HCO3=0.00175,
    T_in=25.0, 
    S_in=35.0, 
    BT=416e-6, 
    ST=0.0282, 
    FT=6.8e-5, 
    Ks=Ks
)

# You can now access any calculated variable using dot notation!
print(f"Calculated pH: {results.pHtot}")
print(f"Calculated TA: {results.TA}")
print(f"Calculated DIC: {results.DIC}")
print(f"Calculated CO2: {results.CO2}")
print(f"Calculated HCO3: {results.HCO3}")
print(f"Calculated CO3: {results.CO3}")

Calculated pH: [8.09003805]
Calculated TA: 0.00232166859469291
Calculated DIC: 0.001992849922247567
Calculated CO2: 1e-05
Calculated HCO3: 0.00175
Calculated CO3: 0.0002328499222991931


In [111]:
# CO2 & CO3

results = calc_C_species(
    CO2=0.00001,
    CO3=0.0002625,
    T_in=25.0, 
    S_in=35.0, 
    BT=416e-6, 
    ST=0.0282, 
    FT=6.8e-5, 
    Ks=Ks
)

# You can now access any calculated variable using dot notation!
print(f"Calculated pH: {results.pHtot}")
print(f"Calculated TA: {results.TA}")
print(f"Calculated DIC: {results.DIC}")
print(f"Calculated CO2: {results.CO2}")
print(f"Calculated HCO3: {results.HCO3}")
print(f"Calculated CO3: {results.CO3}")

Calculated pH: [8.11606465]
Calculated TA: 0.002494096825239711
Calculated DIC: 0.0021305810489395376
Calculated CO2: 1e-05
Calculated HCO3: 0.0018580810489440642
Calculated CO3: 0.0002625


In [113]:
# HCO3 & CO3

results = calc_C_species(
    HCO3=0.00175,
    CO3=0.0002625,
    T_in=25.0, 
    S_in=35.0, 
    BT=416e-6, 
    ST=0.0282, 
    FT=6.8e-5, 
    Ks=Ks
)

# You can now access any calculated variable using dot notation!
print(f"Calculated pH: {results.pHtot}")
print(f"Calculated TA: {results.TA}")
print(f"Calculated DIC: {results.DIC}")
print(f"Calculated CO2: {results.CO2}")
print(f"Calculated HCO3: {results.HCO3}")
print(f"Calculated CO3: {results.CO3}")

Calculated pH: [8.14209126]
Calculated TA: 0.002391232927468278
Calculated DIC: 0.002021370473229768
Calculated CO2: 8.870473230956756e-06
Calculated HCO3: 0.00175
Calculated CO3: 0.0002625


In [7]:
# Moving on to boron! Yay!
# Function chiB
def chiB_calc(H, Ks):
    return 1 / (1 + Ks.KB / H)

In [58]:
# Test chiB

Ks = SimpleNamespace(
    K0 = 0.03,            # Henry's constant for CO2
    K1 = 10.0**(-5.847),   # Carbonate 1
    K2 = 10.0**(-8.966),   # Carbonate 2
    KB = 10.0**(-8.6),     # Boric acid
    KW = 10.0**(-13.2),    # Water
    KP1 = 10.0**(-1.6),    # Phosphoric acid 1
    KP2 = 10.0**(-6.0),    # Phosphoric acid 2
    KP3 = 10.0**(-8.6),    # Phosphoric acid 3
    KSi = 10.0**(-9.5),    # Silicic acid
    KS = 10.0**(-1.0),     # Bisulfate
    KF = 10.0**(-2.6)      # Hydrogen fluoride
)

H_val = 10**(-8.1)

results = chiB_calc(H_val, Ks)
print(f"Calculated value: {results}")

Calculated value: 0.7597469266479578


In [11]:
# Function 1
def BT_BO3(BT, BO3, Ks):
    """
    Returns H
    """
    return Ks.KB / (BT / BO3 - 1)

In [16]:
# Test 1

BT_val = 416 # umol

BO3_val = 316 # umol

results = BT_BO3(BT_val, BO3_val, Ks)
print(f"Calculated value: {results}")


Calculated value: 7.937561123570282e-09


In [18]:
# Function 2
def BT_BO4(BT, BO4, Ks):
    """
    Returns H
    """
    return Ks.KB * (BT / BO4 - 1)

In [20]:
# Test 2

BT_val = 416 # umol

BO4_val = 100 # umol

results = BT_BO4(BT_val, BO4_val, Ks)
print(f"Calculated value: {results}")

Calculated value: 7.937561123570279e-09


In [22]:
# Funciton 3

def pH_BO3(pH, BO3, Ks):
    """
    Returns BT
    """
    H = 10.0**-pH
    return BO3 * (1 + Ks.KB / H)

In [24]:
# Test 3

pH_val = 8.1

BO3_val = 316 # umol

results = pH_BO3(pH_val, BO3_val, Ks)
print(f"Calculated value: {results}")

Calculated value: 415.9279740613208


In [26]:
# Function 4

def pH_BO4(pH, BO4, Ks):
    """
    Returns BT
    """
    H = 10.0**-pH
    return BO4 * (1 + H / Ks.KB)


In [28]:
# Test 4

pH_val = 8.1

BO4_val = 100 # umol

results = pH_BO4(pH_val, BO4_val, Ks)
print(f"Calculated value: {results}")

Calculated value: 416.227766016838


In [30]:
# cBO4

def cBO4(BT, H, Ks):
    return BT / (1 + H / Ks.KB)

In [36]:
# Test cBO4
H_val = 10**(-8.1)

BT_val = 416 # umol

results = cBO4(BT_val, H_val, Ks)
print(f"Calculated value: {results}")

Calculated value: 99.94527851444951


In [34]:
# cBO3
def cBO3(BT, H, Ks):
    return BT / (1 + Ks.KB / H)

In [38]:
# Tst cBO3
H_val = 10**(-8.1)

BT_val = 416 # umol

results = cBO3(BT_val, H_val, Ks)
print(f"Calculated value: {results}")

Calculated value: 316.05472148555043


In [40]:
# calc B function

def calc_B_species(pHtot=None, BT=None, BO3=None, BO4=None, Ks=None, **kwargs):
    # B system calculations
    if pHtot is not None and BT is not None:
        H = 10.0**-pHtot
    elif BT is not None and BO3 is not None:
        H = BT_BO3(BT, BO3, Ks)
    elif BT is not None and BO4 is not None:
        H = BT_BO4(BT, BO4, Ks)
    elif BO3 is not None and BO4 is not None:
        BT = BO3 + BO3
        H = BT_BO3(BT, BO3, Ks)
    elif pHtot is not None and BO3 is not None:
        H = 10.0**-pHtot
        BT = pH_BO3(pHtot, BO3, Ks)
    elif pHtot is not None and BO4 is not None:
        H = 10.0**-pHtot
        BT = pH_BO4(pHtot, BO4, Ks)

    # The above makes sure that BT and H are known,
    # this next bit calculates all the missing species
    # from BT and H.

    if BO3 is None:
        BO3 = cBO3(BT, H, Ks)
    if BO4 is None:
        BO4 = cBO4(BT, H, Ks)
    if pHtot is None:
        pHtot = np.array(-np.log10(H), ndmin=1)

    return Bunch({"Htot": pHtot, "H": H, "BT": BT, "BO3": BO3, "BO4": BO4})

In [44]:
# pH and BT
results = calc_B_species(pHtot=8.1, BT=416, BO3=None, BO4=None, Ks=Ks)

print(f"Calculated values. pH: {results.Htot}, BT: {results.BT}, BO4: {results.BO4}, BO3: {results.BO3}")

Calculated values. pH: 8.1, BT: 416, BO4: 99.94527851444951, BO3: 316.05472148555043


In [46]:
# pH and BO4

results = calc_B_species(pHtot=8.1, BT=None, BO3=None, BO4=100, Ks=Ks)

print(f"Calculated values. pH: {results.Htot}, BT: {results.BT}, BO4: {results.BO4}, BO3: {results.BO3}")

Calculated values. pH: 8.1, BT: 416.227766016838, BO4: 100, BO3: 316.22776601683796


In [48]:
# pH and BO3

results = calc_B_species(pHtot=8.1, BT=None, BO3=316, BO4=None, Ks=Ks)

print(f"Calculated values. pH: {results.Htot}, BT: {results.BT}, BO4: {results.BO4}, BO3: {results.BO3}")

Calculated values. pH: 8.1, BT: 415.9279740613208, BO4: 99.92797406132078, BO3: 316


In [50]:
# BT and BO4

results = calc_B_species(pHtot=None, BT=416, BO3=None, BO4=100, Ks=Ks)

print(f"Calculated values. pH: {results.Htot}, BT: {results.BT}, BO4: {results.BO4}, BO3: {results.BO3}")

Calculated values. pH: [8.10031292], BT: 416, BO4: 100, BO3: 316.0


In [52]:
# BT and BO3

results = calc_B_species(pHtot=None, BT=416, BO3=316, BO4=None, Ks=Ks)

print(f"Calculated values. pH: {results.Htot}, BT: {results.BT}, BO4: {results.BO4}, BO3: {results.BO3}")

Calculated values. pH: [8.10031292], BT: 416, BO4: 99.99999999999997, BO3: 316


In [56]:
# BO4 and BO3

results = calc_B_species(pHtot=None, BT=None, BO3=316, BO4=100, Ks=Ks)

print(f"Calculated values. pH: {results.Htot}, BT: {results.BT}, BO4: {results.BO4}, BO3: {results.BO3}")

Calculated values. pH: [8.6], BT: 632, BO4: 100, BO3: 316


In [62]:
# Defining variables:

In [86]:
Ks = SimpleNamespace(
    K0 = 0.03,            # Henry's constant for CO2
    K1 = 10.0**(-5.847),   # Carbonate 1
    K2 = 10.0**(-8.966),   # Carbonate 2
    KB = 10.0**(-8.6),     # Boric acid
    KW = 10.0**(-13.2),    # Water
    KP1 = 10.0**(-1.6),    # Phosphoric acid 1
    KP2 = 10.0**(-6.0),    # Phosphoric acid 2
    KP3 = 10.0**(-8.6),    # Phosphoric acid 3
    KSi = 10.0**(-9.5),    # Silicic acid
    KS = 10.0**(-1.0),     # Bisulfate
    KF = 10.0**(-2.6)      # Hydrogen fluoride
)

alphaB = 1.0272
A11 = 1.0272
d11 = 27.2
R11 = 4.204
ABT = 0.8078
ABO3 = 0.75
ABO4 = 0.25
d11BT = 39.6
d11B4 = 19.2

In [76]:
# Function get alpha B
def get_alphaB():
    """
    Klochko alpha for B fractionation
    """
    return 1.0272

In [78]:
# test
result = get_alphaB()
print(result)

1.0272


In [80]:
# alpha to epsilon
def alpha_to_epsilon(alphaB):
    """
    Convert alpha to epsilon (which is alpha in delta space)

    Parameters
    ----------
    alphaB : array-like
        The isotope fractionation factor for (11/10 BO3)/(11/10 BO4).

    Returns
    -------
    array-like
        alphaB expressed in delta notation (AKA epsilonB).
    """
    return (alphaB-1)*1000

In [88]:
# test
result = alpha_to_epsilon(alphaB)
print(result)

27.19999999999989


In [90]:
# epsilon to alpha
def epsilon_to_alpha(epsilonB):
    """
    Convert epsilon to alpha

    Parameters
    ----------
    epsilonB : array-like
        The isotope fractionation factor (11/10 BO3)/(11/10 BO4) expressed in delta notation.

    Returns
    -------
    array-like
        The isotope fractionation factor (11/10 BO3)/(11/10 BO4).
    """
    return (epsilonB/1000)+1

In [92]:
# test
result = epsilon_to_alpha(epsilonB=27.2)
print(result)

1.0272


In [94]:
# A11 to d11
def A11_to_d11(A11, SRM_ratio=4.04367):
    """
    Convert fractional abundance (A11) to delta notation (d11).

    Parameters
    ----------
    A11 : array-like
        The fractional abundance of 11B: 11B / (11B + 10B).
    SRM_ratio : float, optional
        The 11B/10B of the SRM, by default NIST951 which is 4.04367

    Returns
    -------
    array-like
        A11 expressed in delta notation (d11).
    """
    return ((A11 / (1 - A11)) / SRM_ratio - 1) * 1000

In [96]:
# test
result = A11_to_d11(A11, SRM_ratio=4.04367)
print(result)

-10339.215584445094


In [100]:
# a11 to r11
def A11_to_R11(A11):
    """
    Convert fractional abundance (A11) to isotope ratio (R11).

    Parameters
    ----------
    A11 : array-like
        The fractional abundance of 11B: 11B / (11B + 10B).

    Returns
    -------
    array-like
        A11 expressed as an isotope ratio (R11).
    """
    return A11 / (1 - A11)


In [104]:
# test
result = A11_to_R11(A11)
print(result)

-37.76470588235309


In [106]:
# d11 to A11
def d11_to_A11(d11, SRM_ratio=4.04367):
    """
    Convert delta notation (d11) to fractional abundance (A11).

    Parameters
    ----------
    d11 : array-like
        The isotope ratio expressed in delta notation.
    SRM_ratio : float, optional
        The 11B/10B of the SRM, by default NIST951 which is 4.04367

    Returns
    -------
    array-like
       Delta notation (d11) expressed as fractional abundance (A11).
    """
    return SRM_ratio * (d11 / 1000 + 1) / (SRM_ratio * (d11 / 1000 + 1) + 1)

In [108]:
# test
result = A11_to_d11(d11, SRM_ratio=4.04367)
print(result)

-1256.7390363039758


In [110]:
# d11 to R11
def d11_to_R11(d11, SRM_ratio=4.04367):
    """
    Convert delta notation (d11) to isotope ratio (R11).

    Parameters
    ----------
    d11 : array-like
        The isotope ratio expressed in delta notation.
    SRM_ratio : float, optional
        The 11B/10B of the SRM, by default NIST951 which is 4.04367

    Returns
    -------
    array-like
       Delta notation (d11) expressed as isotope ratio (R11).
    """
    return (d11 / 1000 + 1) * SRM_ratio

In [112]:
# test
result = d11_to_R11(d11, SRM_ratio=4.04367)
print(result)

4.153657823999999


In [114]:
# R11 to d11
def R11_to_d11(R11, SRM_ratio=4.04367):
    """
    Convert isotope ratio (R11) to delta notation (d11).

    Parameters
    ----------
    R11 : array-like
        The isotope ratio (11B/10B).
    SRM_ratio : float, optional
        The 11B/10B of the SRM, by default NIST951 which is 4.04367

    Returns
    -------
    array-like
        R11 expressed in delta notation (d11).
    """
    return (R11 / SRM_ratio - 1) * 1000


In [116]:
# test
result = R11_to_d11(R11, SRM_ratio=4.04367)
print(result)

39.649625216696684


In [118]:
# R11 to A11
def R11_to_A11(R11):
    """
    Convert isotope ratio (R11) to fractional abundance (A11).

    Parameters
    ----------
    R11 : array-like
        The isotope ratio (11B/10B).

    Returns
    -------
    array-like
        R11 expressed as fractional abundance (A11).
    """
    return R11 / (1 + R11)

In [122]:
# test
result = R11_to_A11(R11)
print(result)

0.8078401229823213


In [124]:
# ABO3 to ABO4
def ABO3_to_ABO4(ABO3,alphaB):
    """
    Converts isotope fractional abundance of boric acid to isotope fraction abundance of borate ion

    Parameters
    ----------
    ABO3 : array-like
        The fractional abundance of boric acid (B(OH)3)
    alphaB : array-like
        The isotope fractionation factor for (11/10 BO3)/(11/10 BO4).

    Returns
    -------
    array-like
        ABO4 - the fractional abundance of borate ion (B(OH)4)
    """
    return (1 / ((alphaB / ABO3) - alphaB + 1) )

In [126]:
# test
result = ABO3_to_ABO4(ABO3,alphaB)
print(result)

0.7449344457687723


In [128]:
# ABO3 or ABO4
def ABO3_or_ABO4(ABO3,ABO4,alphaB):
    """
    Helper function to determine ABO4 if ABO3 is None

    Parameters
    ----------
    ABO3 : array-like
        The fractional abundance of boric acid (B(OH)3)
    ABO4 : array-like
        The fractional abundance of borate ion (B(OH)4)
    alphaB : array-like
        The isotope fractionation factor for (11/10 BO3)/(11/10 BO4).

    Returns
    -------
    array-like
        ABO4 - the fractional abundance of borate ion (B(OH)4)
    """
    if NnotNone(ABO3, ABO4) < 1:
        raise(ValueError("Either ABO4 or ABO3 must be specified"))
    elif ABO4 is None:
        ABO4 = ABO3_to_ABO4(ABO3,alphaB)
    return ABO4


In [138]:
# test for given ABO3
result = ABO3_or_ABO4(ABO3=ABO3,ABO4=None, alphaB=alphaB)
print(result)

0.7449344457687723


In [140]:
# test for given ABO4
result = ABO3_or_ABO4(ABO3=None,ABO4=ABO4, alphaB=alphaB)
print(result)

0.25


In [146]:
# calc ABT

def calculate_ABT(H, Ks, alphaB, ABO4=None, ABO3=None):
    """
    Calculate ABT from pH (total scale) and ABO4 or ABO3.

    Parameters
    ----------    
    Ks : dict
        A dictionary of stoichiometric equilibrium constants.
    pH : array-like
        pH on the Total scale
    alphaB : array-like
        The fractionation factor between B(OH)3 and B(OH)4-
    ABO4 : array-like
        The fractional abundance of 11B in B(OH)4.
    ABO3 : array-like
        The fractional abundance of 11B in B(OH)3.

    Returns
    -------
    array-like
        The fractional abundance of 11B in total B (ABT).
    """
    ABO4 = ABO3_or_ABO4(ABO3,ABO4,alphaB)

    chiB = chiB_calc(H, Ks)
    return (
        ABO4
        * (
            -ABO4 * alphaB * chiB
            + ABO4 * alphaB
            + ABO4 * chiB
            - ABO4
            + alphaB * chiB
            - chiB
            + 1
        )
        / (ABO4 * alphaB - ABO4 + 1))

In [148]:
result = calculate_ABT(H=10**(-8.1), Ks=Ks, alphaB=alphaB, ABO4=None, ABO3=ABO3)
print(result)

0.7487829850277161


In [150]:
result = calculate_ABT(H=10**(-8.1), Ks=Ks, alphaB=alphaB, ABO4=ABO4, ABO3=None)
print(result)

0.25384853925894374
